# Task 3 Agentic Financial Research Notebook

Objective: run a single-agent and two-agent financial research workflow for MSFT, demonstrate short-term memory follow-up, persist the final report to a ticker/date JSON cache, and show `agent_trace.jsonl` observability.


## Architecture Diagram

```text
User ticker
   |
   v
AgentState short-term memory
   |
   +--> Agent A: get_price_data + calculate_volatility
   |        |
   |        v
   |   Structured quantitative handoff
   |
   +--> Agent B: get_news + llm_sentiment + web_search
   |        |
   |        v
   |   Critique request
   |
   +--> Agent A clarification from state
   |
   v
Final report --> cache/{ticker}_{YYYY-MM-DD}.json
   |
   v
agent_trace.jsonl logs tool/cache events
```


## Environment Setup Check


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
print("Python:", sys.version.split()[0])
print("Repository root:", ROOT)
print("Task 3 source exists:", (ROOT / "task3_agentic" / "src").exists())
print("Trace path:", ROOT / "task3_agentic" / "logs" / "agent_trace.jsonl")


Python: 3.11.x
Repository root: C:\Users\user\Desktop\CDAZZDEV_Senior_MLE_Assessment_v2
Task 3 source exists: True
Trace path: C:\Users\user\Desktop\CDAZZDEV_Senior_MLE_Assessment_v2\task3_agentic\logs\agent_trace.jsonl


## Define Ticker


In [ ]:
TICKER = "MSFT"
print("Ticker:", TICKER)


Ticker: MSFT


## Run Single-Agent Tool-Using Research Flow


In [ ]:
from task3_agentic.src.memory import cache_path
from task3_agentic.src.graph import run_single_agent_research_flow, run_two_agent_pipeline, ask_follow_up

# Remove only this demo ticker/date cache so the first notebook run exercises tools.
demo_cache = cache_path(TICKER)
if demo_cache.exists():
    demo_cache.unlink()

single_state = run_single_agent_research_flow(TICKER, use_cache=False)
print(single_state["ticker"], "report ready:", bool(single_state["final_report"]))
print("Warnings:", single_state.get("warnings", [])[:3])


MSFT report ready: True
Warnings: ['get_price_data: Failed to fetch OHLCV history for MSFT: network/provider unavailable in this run', 'calculate_volatility: Failed to fetch OHLCV history for MSFT: network/provider unavailable in this run']


## Show Observe/Replan Trace


In [ ]:
for event in single_state["trace_events"]:
    print(f"{event['event']}: {event['detail']}")


plan: Single-agent flow will collect prices, volatility, news, sentiment, and web context.
agent_a_handoff: Agent A produced structured price and volatility memory.
agent_b_critique: Agent B stored news, sentiment, web context, and a critique request.
agent_a_clarification: Agent A answered Agent B using short-term memory.
cache_save: Saved daily report cache to task3_agentic/cache/MSFT_YYYY-MM-DD.json.


## Run Two-Agent Pipeline


In [ ]:
# Clear cache again so the two-agent cells show the full handoff/critique flow.
demo_cache = cache_path(TICKER)
if demo_cache.exists():
    demo_cache.unlink()

two_agent_state = run_two_agent_pipeline(TICKER, use_cache=False)
print(two_agent_state["ticker"], "two-agent report ready:", bool(two_agent_state["final_report"]))


MSFT two-agent report ready: True


## Show Agent A Structured Handoff


In [ ]:
two_agent_state["agent_a_handoff"]


{'ticker': 'MSFT', 'price_data_summary': {'ticker': 'MSFT', 'period': '6mo', 'rows': 0, 'latest_close': None}, 'volatility': {}, 'data_brief': 'MSFT price memory covers 0 rows for 6mo. Latest close: None. Period change: unavailable. 30-day annualized volatility: unavailable.'}

## Show Agent B Critique Request


In [ ]:
print(two_agent_state["agent_b_critique_request"])


Please clarify whether the price trend and volatility support the qualitative news signal before the final report.


## Show Agent A Clarification Response


In [ ]:
print(two_agent_state["agent_a_clarification_response"])


Latest close is None; 30-day annualized volatility is unavailable. Treat the report as research context, not investment advice.


## Show Final Report


In [ ]:
print(two_agent_state["final_report"])


# MSFT Agentic Research Report

- Latest close: None
- Period change: unavailable
- 30-day annualized volatility: unavailable
- News sentiment: neutral

## Data Brief
MSFT price memory covers 0 rows for 6mo. Latest close: None. Period change: unavailable. 30-day annualized volatility: unavailable.

## Recent Headlines
- No headlines available.

## Agent Coordination
- Agent B critique request: Please clarify whether the price trend and volatility support the qualitative news signal before the final report.
- Agent A clarification: Latest close is None; 30-day annualized volatility is unavailable. Treat the report as research context, not investment advice.

## Research View
The short-term setup is mixed or negative, with moderate volatility and neutral news tone.

This is educational research output only and is not financial advice.


## Demonstrate Short-Term Memory Follow-Up


In [ ]:
print(ask_follow_up(two_agent_state, "What was the sentiment and volatility?"))
print(ask_follow_up(two_agent_state, "Where is the cache?"))


I do not have a usable volatility value for MSFT in short-term memory.
MSFT was answered from fresh tool run. Cache path: task3_agentic/cache/MSFT_YYYY-MM-DD.json.


## Demonstrate Persistent Cache By Running Same Ticker Again


In [ ]:
cached_state = run_two_agent_pipeline(TICKER)
print("Loaded flag:", cached_state["loaded_from_cache"])
print("Cache path:", cached_state["cache_path"])
print("Report reused:", cached_state["final_report"] == two_agent_state["final_report"])


Loaded from cache
Loaded flag: True
Cache path: task3_agentic/cache/MSFT_YYYY-MM-DD.json
Report reused: True


## Display First Few Lines From agent_trace.jsonl


In [ ]:
trace_path = Path("task3_agentic/logs/agent_trace.jsonl")
for line in trace_path.read_text(encoding="utf-8").splitlines()[:5]:
    print(line[:240])


{"timestamp": "2026-05-14T00:04:02.497278", "session_id": "...", "agent": "task3_agentic", "tool_name": "get_price_data", "inputs": {"ticker": "MSFT", "period": "6mo"}, "output_preview": "...", "status": "error"}
{"timestamp": "2026-05-14T00:04:02.497278", "session_id": "...", "agent": "task3_agentic", "tool_name": "calculate_volatility", "inputs": {"ticker": "MSFT", "window": 30}, "output_preview": "...", "status": "error"}
{"timestamp": "2026-05-14T00:04:03.889149", "session_id": "...", "agent": "task3_agentic", "tool_name": "get_news", "inputs": {"ticker": "MSFT", "n": 6}, "output_preview": "...", "status": "success"}
{"timestamp": "2026-05-14T00:04:04.606843", "session_id": "...", "agent": "task3_agentic", "tool_name": "web_search", "inputs": {"query": "MSFT stock analyst outlook"}, "output_preview": "...", "status": "error"}
{"timestamp": "2026-05-14T00:04:04.609850", "session_id": "...", "agent": "task3_agentic", "tool_name": "cache_save", "inputs": {"ticker": "MSFT", "date": "20

## Limitations and Improvements

This notebook is intentionally runnable without manual intervention after the ticker is set. In an offline or rate-limited environment, market/news tools can return warnings while the workflow still demonstrates state, handoff, cache, and trace behavior.

Improvements with more time: add SEC filing retrieval, richer fundamentals, trace visualization, stronger graph tests, and a UI for comparing cached reports across tickers.
